In [ ]:
# ============================================================
# STEP 8O-2: LOAD AND VERIFY DEVELOPMENT DATA
# ============================================================

import numpy as np
import pandas as pd

from pathlib import Path

# Define project directory.
PROJECT_ROOT = Path(
    Path.cwd().parent if Path.cwd().name.lower() == "notebooks" else Path.cwd()
)

# Development dataset location.
BASELINE_DATA_DIR = (
    PROJECT_ROOT / "data" / "processed" / "baseline"
)

DEVELOPMENT_CSV = (
    BASELINE_DATA_DIR / "paderborn_development_features.csv"
)

# Verify file availability.
assert DEVELOPMENT_CSV.exists(), (
    f"Development dataset not found: {DEVELOPMENT_CSV}"
)

# Load development data.
development_df = pd.read_csv(DEVELOPMENT_CSV)

print("DEVELOPMENT DATA LOADED")
print("-" * 50)

print("Shape:", development_df.shape)

print(
    "Unique physical bearings:",
    development_df["bearing"].nunique()
)

print("\nFault class distribution:")
print(
    development_df["fault_class"]
    .value_counts()
    .sort_index()
)


# ============================================================
# DEFINE FINAL 34 FEATURES
# ============================================================

FEATURES_22 = [
    "mean", "std", "min", "max", "kurtosis", "skewness",
    "crest_factor", "dominant_frequency", "dominant_magnitude",
    "spectral_centroid", "spectral_bandwidth",
    "band_power_0_500", "band_power_500_1000",
    "band_power_1000_2000", "band_power_2000_3000",
    "band_power_3000_4000", "band_power_4000_5000",
    "band_power_5000_6000", "band_power_6000_7000",
    "band_power_7000_8000", "band_power_8000_9000",
    "band_power_9000_10000"
]

BPFO_INTEGRATED_FEATURES = [
    "bpfo_1x_energy_ratio",
    "bpfo_2x_energy_ratio",
    "bpfo_3x_energy_ratio",
    "bpfo_4x_energy_ratio",
    "bpfo_5x_energy_ratio",
    "bpfo_1to5_total"
]

BPFI_INTEGRATED_FEATURES = [
    "bpfi_1x_energy_ratio",
    "bpfi_2x_energy_ratio",
    "bpfi_3x_energy_ratio",
    "bpfi_4x_energy_ratio",
    "bpfi_5x_energy_ratio",
    "bpfi_1to5_total"
]

UNIFIED_FAULT_FEATURES = (
    BPFO_INTEGRATED_FEATURES +
    BPFI_INTEGRATED_FEATURES
)

FEATURES_FINAL = (
    FEATURES_22 +
    UNIFIED_FAULT_FEATURES
)


# ============================================================
# VERIFY FEATURE AND LABEL STRUCTURE
# ============================================================

missing_features = [
    feature for feature in FEATURES_FINAL
    if feature not in development_df.columns
]

assert len(FEATURES_FINAL) == 34
assert not missing_features, (
    f"Missing features: {missing_features}"
)

assert len(development_df) == 1999
assert development_df["bearing"].nunique() == 25

assert not development_df[FEATURES_FINAL].isna().any().any()

assert not development_df[FEATURES_FINAL].isin(
    [np.inf, -np.inf]
).any().any()

assert set(development_df["fault_class"].unique()) == {0, 1, 2}


# Create binary target.
# Healthy = 0
# Outer = 1
# Inner = 1

development_df["target"] = (
    development_df["fault_class"]
    .map({
        0: 0,
        1: 1,
        2: 1
    })
)

assert not development_df["target"].isna().any()

print("\nFEATURE VERIFICATION")
print("-" * 50)

print("Total selected features:", len(FEATURES_FINAL))
print("Missing features:", len(missing_features))
print("Missing feature values:", development_df[FEATURES_FINAL].isna().sum().sum())
print("Infinite feature values:", np.isinf(development_df[FEATURES_FINAL]).sum().sum())

print("\nBinary target distribution:")
print(
    development_df["target"]
    .value_counts()
    .sort_index()
    .rename(index={0: "Healthy", 1: "Damaged"})
)

print("\nAll development-data checks passed.")

In [ ]:
# ============================================================
# STEP 8O-3: BUILD AND TRAIN FINAL BASELINE PIPELINE
# ============================================================

import joblib
import json

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression


# ------------------------------------------------------------
# 1. Prepare model inputs
# ------------------------------------------------------------

X_development = development_df[FEATURES_FINAL].copy()

y_development = development_df["target"].astype(int)


print("MODEL TRAINING DATA")
print("-" * 50)

print("Input shape:", X_development.shape)
print("Target shape:", y_development.shape)

print("\nTarget distribution:")
print(y_development.value_counts().sort_index())


# ------------------------------------------------------------
# 2. Create the complete ML pipeline
# ------------------------------------------------------------

final_pipeline = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "classifier",
        LogisticRegression(
            class_weight="balanced",
            max_iter=2000,
            random_state=42
        )
    )
])


# ------------------------------------------------------------
# 3. Train using all development data
# ------------------------------------------------------------

print("\nTraining the final baseline model...")

final_pipeline.fit(
    X_development,
    y_development
)

print("Training completed successfully.")


# ------------------------------------------------------------
# 4. Create model storage directory
# ------------------------------------------------------------

MODEL_DIR = (
    PROJECT_ROOT / "models" / "paderborn_baseline"
)

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# 5. Save the complete pipeline
# ------------------------------------------------------------

MODEL_PATH = MODEL_DIR / "paderborn_baseline_pipeline.joblib"

joblib.dump(
    final_pipeline,
    MODEL_PATH
)


# ------------------------------------------------------------
# 6. Save feature configuration
# ------------------------------------------------------------

FEATURE_CONFIG_PATH = MODEL_DIR / "feature_config.json"

feature_config = {
    "model_name": "Paderborn Binary Bearing Fault Detection",
    "number_of_features": len(FEATURES_FINAL),
    "features": FEATURES_FINAL,
    "target_mapping": {
        "0": "Healthy",
        "1": "Damaged"
    }
}

with open(FEATURE_CONFIG_PATH, "w") as f:
    json.dump(
        feature_config,
        f,
        indent=4
    )


# ------------------------------------------------------------
# 7. Save model metadata
# ------------------------------------------------------------

METADATA_PATH = MODEL_DIR / "model_metadata.json"

model_metadata = {
    "model_version": "paderborn_binary_baseline_v1",
    "dataset": "Paderborn University Bearing Dataset",
    "task": "Binary classification",
    "classifier": "Logistic Regression",
    "scaling": "StandardScaler",
    "class_weight": "balanced",
    "max_iter": 2000,
    "random_state": 42,
    "decision_threshold": 0.5,
    "training_recordings": len(development_df),
    "training_bearings": int(
        development_df["bearing"].nunique()
    ),
    "healthy_recordings": int(
        (y_development == 0).sum()
    ),
    "damaged_recordings": int(
        (y_development == 1).sum()
    ),
    "number_of_features": len(FEATURES_FINAL)
}

with open(METADATA_PATH, "w") as f:
    json.dump(
        model_metadata,
        f,
        indent=4
    )


# ------------------------------------------------------------
# 8. Verify saved artifacts
# ------------------------------------------------------------

print("\nMODEL ARTIFACTS")
print("-" * 50)

for path in [
    MODEL_PATH,
    FEATURE_CONFIG_PATH,
    METADATA_PATH
]:

    assert path.exists(), f"Missing artifact: {path}"

    print(f"\nFile: {path.name}")
    print(f"Size: {path.stat().st_size / 1024:.2f} KB")


print("\nBaseline model successfully trained and saved.")

In [ ]:
# ============================================================
# STEP 8O-4: LOAD AND VERIFY SAVED BASELINE MODEL
# ============================================================

import joblib
import json
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# 1. Load the saved model
# ------------------------------------------------------------

loaded_pipeline = joblib.load(MODEL_PATH)

print("Saved pipeline loaded successfully.")


# ------------------------------------------------------------
# 2. Load feature configuration
# ------------------------------------------------------------

with open(FEATURE_CONFIG_PATH, "r") as f:
    loaded_config = json.load(f)

loaded_features = loaded_config["features"]

print("Feature configuration loaded successfully.")

print("Number of features:", len(loaded_features))


# ------------------------------------------------------------
# 3. Verify feature ordering
# ------------------------------------------------------------

assert loaded_features == FEATURES_FINAL, (
    "Feature ordering mismatch!"
)

print("Feature ordering verified.")


# ------------------------------------------------------------
# 4. Prepare development data for verification
# ------------------------------------------------------------

X_check = development_df[loaded_features]


# ------------------------------------------------------------
# 5. Generate predictions using original pipeline
# ------------------------------------------------------------

original_predictions = final_pipeline.predict(X_check)

original_probabilities = (
    final_pipeline.predict_proba(X_check)[:, 1]
)


# ------------------------------------------------------------
# 6. Generate predictions using loaded pipeline
# ------------------------------------------------------------

loaded_predictions = loaded_pipeline.predict(X_check)

loaded_probabilities = (
    loaded_pipeline.predict_proba(X_check)[:, 1]
)


# ------------------------------------------------------------
# 7. Compare predictions
# ------------------------------------------------------------

predictions_match = np.array_equal(
    original_predictions,
    loaded_predictions
)

probabilities_match = np.allclose(
    original_probabilities,
    loaded_probabilities,
    rtol=1e-10,
    atol=1e-12
)

print("\nPREDICTION CONSISTENCY CHECK")
print("-" * 50)

print("Predictions identical:", predictions_match)
print("Probabilities identical:", probabilities_match)

assert predictions_match
assert probabilities_match


# ------------------------------------------------------------
# 8. Display sample predictions
# ------------------------------------------------------------

prediction_preview = pd.DataFrame({
    "actual_target": development_df["target"].values[:10],
    "predicted_target": loaded_predictions[:10],
    "damaged_probability": loaded_probabilities[:10]
})

prediction_preview["predicted_class"] = (
    prediction_preview["predicted_target"]
    .map({
        0: "Healthy",
        1: "Damaged"
    })
)

print("\nSAMPLE PREDICTIONS")

display(prediction_preview.round(4))


print("\nMODEL VERIFICATION SUCCESSFUL.")